In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════
Version 6 — Soft QPSK Input VGG Network (86.81% Accuracy)
QPSK-Constrained Weights & Activations for Fully Quantized RF Inference
═══════════════════════════════════════════════════════════════════════════════

This model implements a VGG-style CNN on CIFAR-10 using an end-to-end QPSK
signal pipeline designed for wireless / OFDM-style neural transmission.
RGB images are first projected from 3→2 channels using a tiny learnable 1×1
float convolution, then softly quantized to QPSK using β-annealed soft
quantization (β: 0.1 → 10), allowing smooth transition from float-like
training to near-hard constellation assignment without destroying gradients.
All convolution weights, linear weights, and intermediate activations are
QPSK-constrained to {(±1 ± j)/√2} using latent BinaryConnect-style weights
with Straight-Through Estimation (STE). Each layer maintains separate real
and imaginary tensors and performs RF-style magnitude recovery:
sqrt(real² + imag²). Despite using only 2-level QPSK inputs, Version 6
achieved 86.81% accuracy, showing that the learned 1×1 semantic projection
preserves critical image structure even under extreme quantization.
═══════════════════════════════════════════════════════════════════════════════
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import math

# ─────────────────────────────────────────────
# DEVICE
# ─────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ─────────────────────────────────────────────
# QPSK QUANTIZER  (STE)
# ─────────────────────────────────────────────
class QPSKQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        # Snaps every element to the nearest QPSK constellation point
        # (±1/√2) in the forward pass, while recording the input so
        # the backward pass can apply the straight-through estimator.
        ctx.save_for_backward(x)
        return torch.sign(x) / math.sqrt(2)

    @staticmethod
    def backward(ctx, grad_output):
        # Straight-through estimator: passes the upstream gradient
        # back unchanged, bypassing the zero/undefined derivative of
        # the sign function so that latent weights continue to receive
        # useful update signals during training.
        return grad_output.clone()

qpsk_quantize = QPSKQuantize.apply


# ─────────────────────────────────────────────
# QPSK SOFT INPUT QUANTIZER
# ─────────────────────────────────────────────

# QPSK 1D levels: [-1, +1] / sqrt(2)
# These are identical to the weight/activation QPSK constellation levels.
# Average power: (0.5 + 0.5) / 2 = 0.5 per axis → total symbol power = 1.0
# No additional normalization factor needed.
_QPSK_INPUT_LEVELS = torch.tensor([-1., 1.]) / math.sqrt(2)


def soft_qpsk_input_quantize(x: torch.Tensor, beta: float) -> torch.Tensor:
    levels = _QPSK_INPUT_LEVELS.to(x.device)          # (2,)

    # Expand x for broadcasting: (..., 1) vs (2,)
    x_exp = x.unsqueeze(-1)                            # (..., 1)
    dists = (x_exp - levels).abs()                     # (..., 2)

    # Softmax with negative distances: closer point gets higher weight
    weights = F.softmax(-beta * dists, dim=-1)         # (..., 2)

    # Weighted sum — at high β collapses to sign(x)/√2
    out = (weights * levels).sum(dim=-1)               # (...,)
    return out


def input_snap_error(x_soft: torch.Tensor) -> float:
    # Finds the nearest hard QPSK level for every element in the soft-
    # quantized tensor, then returns the mean absolute deviation as a
    # scalar. Values close to zero confirm that β is large enough for the
    # soft quantizer to have effectively collapsed to the two-point QPSK
    # grid, validating the annealing schedule at a given training epoch.
    levels = _QPSK_INPUT_LEVELS.to(x_soft.device)
    dists = (x_soft.unsqueeze(-1) - levels).abs()      # (..., 2)
    nearest_idx = dists.argmin(dim=-1)                 # (...,)
    nearest = levels[nearest_idx]                      # (...,)
    return (x_soft - nearest).abs().mean().item()


# ─────────────────────────────────────────────
# SOFT QPSK INPUT PROJECTION MODULE
# ─────────────────────────────────────────────
class SoftQPSKInput(nn.Module):
    # Max constellation value: 1 / sqrt(2)  ≈ 0.707
    _MAX_VAL = 1.0 / math.sqrt(2)

    def __init__(self):
        super().__init__()
        # Float 1×1 conv: 3 RGB channels → 2 complex channels
        # NOT QPSK-quantized — this is a float preprocessing layer
        self.proj = nn.Conv2d(3, 2, kernel_size=1, bias=True)
        self._init_proj()

    def _init_proj(self):
        # Sets the 1×1 projection weights to physically meaningful starting
        # values: the real channel approximates perceptual luminance (standard
        # Rec.601 luma coefficients) and the imaginary channel approximates the
        # Cb chrominance component, giving the optimizer a sensible starting
        # point rather than random noise.
        with torch.no_grad():
            self.proj.weight.data = torch.tensor([
                [[ 0.299],  [ 0.587],  [ 0.114]],   # real ≈ luminance
                [[ 0.500],  [-0.419],  [-0.081]],    # imag ≈ Cb-like
            ]).view(2, 3, 1, 1)
            self.proj.bias.data.zero_()

    def forward(self, x: torch.Tensor, beta: float) -> torch.Tensor:
        # Step 1: learned RGB → complex projection (float)
        x = self.proj(x)                               # (B, 2, H, W)

        # Step 2: Tanh bounding — maps ℝ → [-1/√2, 1/√2]
        x = torch.tanh(x) * self._MAX_VAL             # (B, 2, H, W)

        # Step 3: soft QPSK quantization (independent per channel/pixel)
        x = soft_qpsk_input_quantize(x, beta)         # (B, 2, H, W)

        return x

    def get_snap_error(self, x_raw: torch.Tensor, beta: float) -> float:
        # Convenience wrapper for use in the diagnostics loop: runs the full
        # forward pass inside a no-grad context and returns the scalar snap
        # error, so callers get a single float without having to manage
        # gradient tapes.
        with torch.no_grad():
            x_soft = self.forward(x_raw, beta)
        return input_snap_error(x_soft)


# ─────────────────────────────────────────────
# DIAGNOSTICS HELPERS  (QPSK weights)
# ─────────────────────────────────────────────
def sign_commitment(w: torch.Tensor) -> float:
    # Returns the fraction of latent weight values whose magnitude exceeds
    # 0.1, used as a proxy for how firmly each weight has "committed" to a
    # QPSK sign. A value close to 1.0 means nearly all weights are well away
    # from zero and will round stably to their intended constellation point.
    return (w.abs() > 0.1).float().mean().item()


def qpsk_snap_error(w_real: torch.Tensor, w_imag: torch.Tensor) -> float:
    # Computes the mean absolute difference between the latent (real-valued)
    # weight tensor and its hard-quantized QPSK counterpart, averaged over
    # both the real and imaginary components. Small values indicate that
    # latent weights have converged close to the discrete QPSK grid.
    q_real = torch.sign(w_real) / math.sqrt(2)
    q_imag = torch.sign(w_imag) / math.sqrt(2)
    err = ((w_real - q_real).abs().mean() + (w_imag - q_imag).abs().mean()) / 2.0
    return err.item()


def sign_penalty(w_real: torch.Tensor, w_imag: torch.Tensor) -> torch.Tensor:
    # Penalizes latent weights near zero by adding exp(-|w|) for each
    # component. Minimizing this loss pushes weights toward large positive
    # or negative values, encouraging decisive sign commitment and preventing
    # the soft latent weights from stalling near the QPSK decision boundary
    # during training.
    return (torch.exp(-w_real.abs()) + torch.exp(-w_imag.abs())).mean()


# ─────────────────────────────────────────────
# QPSK CONV LAYER
# ─────────────────────────────────────────────
class QPSKConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 padding=0, quantize_input=True):
        super().__init__()
        self.quantize_input = quantize_input
        self.w_real = nn.Parameter(
            torch.Tensor(out_channels, in_channels, kernel_size, kernel_size))
        self.w_imag = nn.Parameter(
            torch.Tensor(out_channels, in_channels, kernel_size, kernel_size))
        self.padding = padding
        self._init_weights()

    def _init_weights(self):
        # Initializes both the real and imaginary weight tensors with Kaiming
        # uniform values scaled down by 0.3 so the initial latent magnitudes
        # sit near the QPSK decision boundary, giving the sign-penalty
        # regularizer room to push weights toward commitment without
        # immediately saturating.
        nn.init.kaiming_uniform_(self.w_real, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_imag, a=math.sqrt(5))
        with torch.no_grad():
            self.w_real.data *= 0.3
            self.w_imag.data *= 0.3

    def forward(self, x):
        # Optionally quantizes the input activations to QPSK, then performs
        # two standard convolutions with QPSK-snapped real and imaginary
        # weight tensors and returns the per-element complex magnitude of
        # the result.
        if self.quantize_input:
            x = qpsk_quantize(x)
        w_q_real = qpsk_quantize(self.w_real)
        w_q_imag = qpsk_quantize(self.w_imag)
        out_real = F.conv2d(x, w_q_real, padding=self.padding)
        out_imag = F.conv2d(x, w_q_imag, padding=self.padding)
        return torch.sqrt(out_real ** 2 + out_imag ** 2 + 1e-8)

    def diagnostics(self):
        # Returns a dictionary of scalar health metrics for this layer:
        # what fraction of real/imaginary latent weights have committed to
        # a sign and how far the latent weights sit from the nearest QPSK
        # point on average.
        return {
            "sign_commit_real": sign_commitment(self.w_real),
            "sign_commit_imag": sign_commitment(self.w_imag),
            "snap_error":       qpsk_snap_error(self.w_real, self.w_imag),
        }

    def penalty(self):
        # Returns the bimodal sign-penalty scalar for this layer's weight
        # tensors, to be scaled by λ_conv and summed into the total
        # training loss.
        return sign_penalty(self.w_real, self.w_imag)


# ─────────────────────────────────────────────
# QPSK LINEAR LAYER
# ─────────────────────────────────────────────
class QPSKLinear(nn.Module):
    #FC layer with QPSK-quantized weights and input activations.
    #Output: complex magnitude sqrt(r² + i²).

    def __init__(self, in_features, out_features):
        super().__init__()
        self.w_real = nn.Parameter(torch.Tensor(out_features, in_features))
        self.w_imag = nn.Parameter(torch.Tensor(out_features, in_features))
        self._init_weights()

    def _init_weights(self):
        # Mirrors the QPSKConv2d initialization strategy: Kaiming uniform
        # scaled by 0.3 to place initial latent weights near the QPSK
        # decision boundary, making the sign-penalty regularizer
        # immediately effective.
        nn.init.kaiming_uniform_(self.w_real, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_imag, a=math.sqrt(5))
        with torch.no_grad():
            self.w_real.data *= 0.3
            self.w_imag.data *= 0.3

    def forward(self, x):
        # Quantizes input activations to QPSK, applies two separate linear
        # projections with QPSK-snapped real and imaginary weight matrices,
        # and returns the element-wise complex magnitude of the results.
        x = qpsk_quantize(x)
        w_q_real = qpsk_quantize(self.w_real)
        w_q_imag = qpsk_quantize(self.w_imag)
        out_real = F.linear(x, w_q_real)
        out_imag = F.linear(x, w_q_imag)
        return torch.sqrt(out_real ** 2 + out_imag ** 2 + 1e-8)

    def diagnostics(self):
        # Returns the same set of scalar health metrics as
        # QPSKConv2d.diagnostics() so that all QPSK layers — convolutional
        # and fully-connected alike — can be monitored with a uniform
        # interface during the diagnostic logging pass.
        return {
            "sign_commit_real": sign_commitment(self.w_real),
            "sign_commit_imag": sign_commitment(self.w_imag),
            "snap_error":       qpsk_snap_error(self.w_real, self.w_imag),
        }

    def penalty(self):
        # Returns the bimodal sign-penalty scalar for this layer's weight
        # tensors, to be scaled by λ_fc and summed into the total
        # training loss.
        return sign_penalty(self.w_real, self.w_imag)


# ─────────────────────────────────────────────
# NETWORK  (VGG-style + Soft QPSK input)
# ─────────────────────────────────────────────
class QPSKNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # ── Soft QPSK input quantization (float preprocessing) ───────────
        self.input_qam = SoftQPSKInput()

        # ── Block 1 (conv1 takes 2ch from QPSK input, not 3ch RGB) ──────────
        self.conv1 = QPSKConv2d(2,   128, 3, padding=1, quantize_input=False)
        self.bn1   = nn.BatchNorm2d(128, momentum=0.1, eps=1e-4)
        self.conv2 = QPSKConv2d(128, 128, 3, padding=1, quantize_input=True)
        self.bn2   = nn.BatchNorm2d(128, momentum=0.1, eps=1e-4)
        self.pool1 = nn.MaxPool2d(2, 2)                # 32→16

        # ── Block 2 ───────────────────────────────────────────────────────
        self.conv3 = QPSKConv2d(128, 256, 3, padding=1, quantize_input=True)
        self.bn3   = nn.BatchNorm2d(256, momentum=0.1, eps=1e-4)
        self.conv4 = QPSKConv2d(256, 256, 3, padding=1, quantize_input=True)
        self.bn4   = nn.BatchNorm2d(256, momentum=0.1, eps=1e-4)
        self.pool2 = nn.MaxPool2d(2, 2)                # 16→8

        # ── Block 3 ───────────────────────────────────────────────────────
        self.conv5 = QPSKConv2d(256, 512, 3, padding=1, quantize_input=True)
        self.bn5   = nn.BatchNorm2d(512, momentum=0.1, eps=1e-4)
        self.conv6 = QPSKConv2d(512, 512, 3, padding=1, quantize_input=True)
        self.bn6   = nn.BatchNorm2d(512, momentum=0.1, eps=1e-4)
        self.pool3 = nn.MaxPool2d(2, 2)                # 8→4

        # ── Classifier (4×4×512=8192 → 1024 → 1024 → 10) ────────────────
        self.fc1    = QPSKLinear(8192, 1024)
        self.bn_fc1 = nn.BatchNorm1d(1024, momentum=0.1, eps=1e-4)
        self.fc2    = QPSKLinear(1024, 1024)
        self.bn_fc2 = nn.BatchNorm1d(1024, momentum=0.1, eps=1e-4)
        self.fc_out = nn.Linear(1024, num_classes)     # float logit head

    def forward(self, x: torch.Tensor, beta: float) -> torch.Tensor:
        # Defines the complete data flow from a raw RGB batch to class logits:
        # soft QPSK encoding → three QPSK conv blocks with batch norm and
        # max-pooling → flatten → two QPSK FC layers with batch norm → float
        # linear head. The β parameter is threaded through to the input module
        # so the annealing schedule is applied consistently at each training
        # step. Unlike v5, the input module here targets the two-point QPSK
        # constellation, making the entire signal chain uniform.

        # ── Soft QPSK input quantization ─────────────────────────────────
        x = self.input_qam(x, beta)                    # (B, 2, 32, 32)

        # ── Conv blocks ───────────────────────────────────────────────────
        x = self.bn1(self.conv1(x))
        x = self.bn2(self.conv2(x))
        x = self.pool1(x)

        x = self.bn3(self.conv3(x))
        x = self.bn4(self.conv4(x))
        x = self.pool2(x)

        x = self.bn5(self.conv5(x))
        x = self.bn6(self.conv6(x))
        x = self.pool3(x)

        # ── FC classifier ─────────────────────────────────────────────────
        x = x.view(x.size(0), -1)
        x = self.bn_fc1(self.fc1(x))
        x = self.bn_fc2(self.fc2(x))
        x = self.fc_out(x)
        return x

    def qpsk_layers(self):
        # Yields (name, layer) pairs for every QPSK-constrained layer in the
        # network (all six conv layers and both FC layers), providing a single
        # iteration point for diagnostics, penalty accumulation, and
        # BinaryConnect weight clipping without having to enumerate layers
        # manually elsewhere.
        """(name, layer) pairs for all QPSK weight layers."""
        return [
            ("conv1", self.conv1), ("conv2", self.conv2),
            ("conv3", self.conv3), ("conv4", self.conv4),
            ("conv5", self.conv5), ("conv6", self.conv6),
            ("fc1",   self.fc1),   ("fc2",   self.fc2),
        ]

    def get_all_diagnostics(self):
        # Collects and returns the diagnostics dictionary from every QPSK
        # layer in one call, keyed by layer name. Used during the periodic
        # diagnostic logging pass to print sign-commitment and snap-error
        # statistics for the entire network in a single loop without touching
        # individual layers directly.
        return {name: layer.diagnostics() for name, layer in self.qpsk_layers()}

    def total_sign_penalty(self, lambda_conv: float, lambda_fc: float) -> torch.Tensor:
        # Computes the network-wide bimodal sign-penalty regularization term
        # by summing each layer's penalty weighted by its corresponding λ
        # coefficient (λ_conv for convolutional layers, λ_fc for
        # fully-connected layers). The resulting scalar is added to the
        # cross-entropy loss at every step.
        penalty = torch.tensor(0.0, device=next(self.parameters()).device)
        for name, layer in self.qpsk_layers():
            lam = lambda_fc if name.startswith("fc") else lambda_conv
            penalty = penalty + lam * layer.penalty()
        return penalty


# ─────────────────────────────────────────────
# BETA ANNEALING
# ─────────────────────────────────────────────
def get_beta(epoch: int, beta_start: float, beta_end: float,
             anneal_epochs: int) -> float:
    # Computes the soft-quantizer temperature β for a given epoch using an
    # exponential schedule from beta_start to beta_end over anneal_epochs
    # epochs, then holds it constant at beta_end. Exponential interpolation
    # is chosen because the soft-to-hard transition is inherently nonlinear:
    # it allocates more training time in the soft (informative gradient) regime
    # and less time once the quantizer is already nearly discrete.
    if epoch >= anneal_epochs:
        return beta_end
    # Exponential interpolation
    t = (epoch - 1) / (anneal_epochs - 1)
    return beta_start * (beta_end / beta_start) ** t


# ─────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────
def get_cifar10_loaders(batch_size=128, num_workers=2):
    # Constructs PyTorch DataLoaders for CIFAR-10 with standard augmentation
    # (random crop with padding and horizontal flip) applied to training data
    # only. Normalization uses the dataset's per-channel mean and standard
    # deviation so pixel values are zero-centered before entering the network.
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2023, 0.1994, 0.2010)

    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_set = torchvision.datasets.CIFAR10(
        root='./data', train=True,  download=True, transform=train_transform)
    test_set  = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=test_transform)

    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True)
    test_loader  = torch.utils.data.DataLoader(
        test_set,  batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True)

    return train_loader, test_loader


# ─────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────
def evaluate(model, loader, beta):
    # Runs inference over an entire DataLoader in evaluation mode (BatchNorm
    # uses running statistics, no dropout) with gradients disabled for speed.
    # Returns top-1 classification accuracy as a percentage. The current β is
    # passed through so the soft-quantizer operates at the same temperature
    # used during the most recent training step.
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images, beta)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total


# ─────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, beta,
                    lambda_conv, lambda_fc, log_grads=False):
    # Executes one full pass over the training DataLoader: computes the
    # combined cross-entropy + sign-penalty loss, back-propagates, optionally
    # logs per-layer gradient norms on the first batch of epoch 1 to detect
    # dead weights, applies BinaryConnect-style latent weight clipping to
    # [-1, 1] after each optimizer step, and returns the epoch-averaged CE
    # loss, penalty loss, and training accuracy.
    model.train()
    running_ce  = 0.0
    running_pen = 0.0
    correct = total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs  = model(images, beta)
        ce_loss  = F.cross_entropy(outputs, labels)
        pen_loss = model.total_sign_penalty(lambda_conv, lambda_fc)
        loss     = ce_loss + pen_loss
        loss.backward()

        # ── Gradient check — first batch of epoch 1 only ─────────────────
        if batch_idx == 0 and log_grads:
            print("\n  [GRADIENT CHECK — Batch 0]")
            # Check QPSK weight gradients
            for lname, layer in [("conv5", model.conv5), ("conv6", model.conv6),
                                  ("fc1",   model.fc1),   ("fc2",   model.fc2)]:
                gr = layer.w_real.grad
                gi = layer.w_imag.grad
                if gr is not None:
                    print(f"  {lname}: grad_real={gr.abs().mean().item():.6f}  "
                          f"grad_imag={gi.abs().mean().item():.6f}")
                else:
                    print(f"  {lname}: NO GRADIENT")
            # Check 1×1 proj gradient
            pg = model.input_qam.proj.weight.grad
            if pg is not None:
                print(f"  proj: grad={pg.abs().mean().item():.6f}")
            print()

        optimizer.step()

        # ── BinaryConnect weight clipping ─────────────────────────────────
        with torch.no_grad():
            for _, layer in model.qpsk_layers():
                layer.w_real.clamp_(-1.0, 1.0)
                layer.w_imag.clamp_(-1.0, 1.0)

        running_ce  += ce_loss.item()
        running_pen += pen_loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    n = len(loader)
    return running_ce / n, running_pen / n, 100.0 * correct / total


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    # Entry point that wires together all components: declares hyperparameters,
    # instantiates the model, optimizer (Adam, weight_decay=0), and
    # MultiStepLR scheduler, then runs the training loop for 180 epochs. At
    # each epoch it computes the current β, trains for one epoch, evaluates on
    # the test set, checkpoints the best model, and logs metrics. Every
    # DIAG_EVERY epochs it prints per-layer QPSK health statistics and input
    # snap error.

    # ── Hyperparameters ──────────────────────────────────────────────────
    EPOCHS      = 180
    BATCH_SIZE  = 128
    LR          = 1e-3

    # Sign penalty
    LAMBDA_CONV = 1e-4
    LAMBDA_FC   = 3e-4

    # LR schedule
    LR_MILESTONES = [100, 150]
    LR_GAMMA      = 0.1

    # β annealing for soft QPSK input quantizer
    BETA_START     = 0.1    # nearly float at start
    BETA_END       = 10.0   # nearly hard-quantized at end
    BETA_ANNEAL_EP = 100    # anneal over first 100 epochs, hold at BETA_END after

    DIAG_EVERY = 20

    print("=" * 70)
    print("QPSK-Quantized CNN on CIFAR-10  [DEBUG MODE — v6]")
    print("Input        : Soft QPSK (1×1 Conv proj + Tanh×1/√2 + annealed β)")
    print("Weights/Acts : QPSK-constrained  |  weight_decay: 0")
    print(f"Scheduler    : MultiStepLR  milestones={LR_MILESTONES}  γ={LR_GAMMA}")
    print(f"Sign penalty : λ_conv={LAMBDA_CONV}  λ_fc={LAMBDA_FC}")
    print(f"β schedule   : {BETA_START} → {BETA_END} over {BETA_ANNEAL_EP} epochs")
    print(f"Epochs       : {EPOCHS}  |  grad logging: epoch 1")
    print("=" * 70)

    train_loader, test_loader = get_cifar10_loaders(BATCH_SIZE)

    model     = QPSKNet(num_classes=10).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=LR_MILESTONES, gamma=LR_GAMMA)

    total_params = sum(p.numel() for p in model.parameters())
    proj_params  = sum(p.numel() for p in model.input_qam.proj.parameters())
    print(f"\nTotal parameters : {total_params:,}")
    print(f"  of which proj  : {proj_params}  (float, not QPSK)")
    print()

    best_acc = 0.0

    # Keep a sample batch for input snap-error diagnostics
    sample_images = None

    for epoch in range(1, EPOCHS + 1):
        beta = get_beta(epoch, BETA_START, BETA_END, BETA_ANNEAL_EP)

        ce_loss, pen_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, beta,
            LAMBDA_CONV, LAMBDA_FC,
            log_grads=(epoch == 1)
        )
        test_acc = evaluate(model, test_loader, beta)
        scheduler.step()

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), "qpsk_cifar10_v6_best.pth")

        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch [{epoch:3d}/{EPOCHS}]  "
              f"CE: {ce_loss:.4f}  Pen: {pen_loss:.4f}  "
              f"Train: {train_acc:.2f}%  Test: {test_acc:.2f}%  "
              f"Best: {best_acc:.2f}%  "
              f"LR: {current_lr:.2e}  β: {beta:.3f}")

        # ── Diagnostics ───────────────────────────────────────────────────
        if epoch % DIAG_EVERY == 0:
            print(f"\n  [DIAGNOSTICS — Epoch {epoch}  β={beta:.3f}]")

            # QPSK weight diagnostics
            diag = model.get_all_diagnostics()
            for lname, d in diag.items():
                print(f"  {lname:6s}: "
                      f"commit_r={d['sign_commit_real']:.3f}  "
                      f"commit_i={d['sign_commit_imag']:.3f}  "
                      f"snap_err={d['snap_error']:.4f}")

            # Input quantization snap error (on a fixed sample batch)
            # Grab first batch from test loader as reference
            if sample_images is None:
                sample_images = next(iter(test_loader))[0][:64].to(device)
            inp_snap = model.input_qam.get_snap_error(sample_images, beta)
            print(f"\n  Input QPSK snap_error = {inp_snap:.6f}  "
                  f"(→0 as β→∞, currently β={beta:.3f})")
            print()

    print(f"\nTraining complete. Best test accuracy: {best_acc:.2f}%")
    print("Model saved to: qpsk_cifar10_v6_best.pth")


if __name__ == "__main__":
    main()